In [1]:
import torch
import os
import subprocess
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import snapshot_download, login
from datasets import load_dataset



In [2]:
from peft import PeftModel, PeftConfig
import json
import re
from IPython.display import Markdown, display
import numpy as np
import json
from tqdm import tqdm
from torch.nn.attention import SDPBackend, sdpa_kernel

In [3]:
dataset = load_dataset("trl-lib/DeepMath-103K", split="train")

README.md:   0%|          | 0.00/692 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/12.2M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/641k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/97870 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5152 [00:00<?, ? examples/s]

In [4]:
dataset

Dataset({
    features: ['prompt', 'solution'],
    num_rows: 97870
})

In [6]:
dataset[3]

{'prompt': [{'content': 'Let $p(x)$ be the polynomial of degree at most 8 such that\n\\[p(n) = \\frac{1}{n}\\]for $n = 1,$ 2, 3, $\\dots,$ 9.  Find $p(10).$',
   'role': 'user'}],
 'solution': '$\\frac{1}{5}$'}

In [3]:
dataset = load_dataset("json", data_files="/data/Iman/datasets/ownship_prompts_with_action_rewards_dataNov17.jsonl", split='train')

In [5]:
dataset

Dataset({
    features: ['system_prompt', 'user_prompt', 'action_rewards', 'prompt_response'],
    num_rows: 19854
})

In [4]:
import re

def to_grpo_row(ex):
    prompt = (
        f"### System:\n{ex['system_prompt'].strip()}\n"
        f"### User:\n{ex['user_prompt'].strip()}. Always start your response with: {"The recommended action is:"} \n"
        f"### Assistant: "
    )
    prompt_response = ex['prompt_response']
    action = re.search(r"\b(Accelerate|Hold|Decelerate)\b", prompt_response, flags=re.IGNORECASE)
    action = action.group(1).capitalize() if action else None
    solution = "The recommended action is: " + action + "."

    return {
        "prompt": [{'content': prompt, 'role': 'user' }],
        "solution": solution,       # dict per prompt
    }

ds_train = dataset.map(to_grpo_row, remove_columns=dataset.column_names)
# ds_test = dataset["test"].map(to_grpo_row, remove_columns=ds["test"].column_names)

In [5]:
import random

indices = random.sample(range(len(ds_train)), 1000)
sampled_ds = ds_train.select(indices)
sampled_ds


Dataset({
    features: ['prompt', 'solution'],
    num_rows: 1000
})

In [8]:
sampled_ds['prompt'][50][0]['content']

'### System:\nYou are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, \n    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, \n    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, \n    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by \n    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.\n    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],\n    v: ownship speed(m/s), \n    v_min: ownship minimum speed(m/s), \n    v_max: ownship maximum speed(m/s),\n    d1: th

In [9]:
sampled_ds['solution'][50]

'The recommended action is: Decelerate.'

In [9]:
ds_train[0]

{'prompt': [{'content': '### System:\nYou are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, \n    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, \n    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, \n    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by \n    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.\n    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],\n    v: ownship speed(m/s), \n    v_min: ownship minimum speed(m/s), \n    v_max: ownship maximum

In [ ]:
torch.backends.cuda.flash_sdp_enabled()

In [10]:
BASE_MODEL_DIR = "/data/Iman/shared_LLM_models/Qwen2.5-Math-7B/"
ADAPTER_DIR = "/data/Iman/finetuned_models/grpo-qwen-atc3/grpo_qwen_lora_adapter"

## 1. Load tokenizer and model

In [7]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, padding_side='left', trust_remote_code=True)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

# attach LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_DIR)

# model = model.merge_and_unload()
model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The module name  (originally ) is not a valid Python identifier. Please rename the original module to avoid import issues.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(152064, 3584)
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
              (k_proj): Linear(in_features=3584, out_features=512, bias=True)
              (v_proj): Linear(in_features=3584, out_features=512, bias=True)
              (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
            )
            (mlp): Qwen2MLP(
              (gate_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=18944, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3584, out_features=8, bias=False)
     

In [ ]:
# saving the merged model
# merged_model_dir = "./finetuned_models/merged-qwen-atc-model/"

# model.save_pretrained(merged_model_dir)
# tokenizer.save_pretrained(merged_model_dir)

In [ ]:
# load the merged model
# quantization_config = BitsAndBytesConfig(
#     load_in_16bit=True,
# )
# # model = AutoModelForCausalLM.from_pretrained(
# #     merged_model_dir,
# #     torch_dtype=torch.float16,
# #     device_map="auto",
# #     # quantization_config=quantization_config,
# #     # attn_implementation="sdpa",
# #     # trust_remote_code=True,
# # )
# model = AutoModelForCausalLM.from_pretrained(
#     BASE_MODEL_DIR,
#     torch_dtype=torch.float16,
#     device_map="auto",
#     # quantization_config=quantization_config,
#     # attn_implementation="sdpa",
#     # trust_remote_code=True,
# )

# model.eval()


## 2. Extracting Data and formating

In [8]:
# read the data from json file
file_path = 'ownship_prompts_data.jsonl'
with open(file_path, 'r') as f:
    data = [json.loads(line) for line in f]

# ds = load_dataset('json', data_files=file_path, split='train').train_test_split(test_size=0.2, seed=42)
# test = ds['test']

def format_prompt(ex):
    # return (f"### System:\n{example['system_prompt'].strip()}\n\n"
    #         f"### User:\n{example['user_prompt'].strip()}\n\n"
    #         f"### Assistant:\n{example['prompt_response'].strip()}"
    # )
    return (
        f"### System:\n{ex['system_prompt'].strip()}\n"
        f"### User:\n{ex['user_prompt'].strip()}. Always start your response with: {"The recommended action is:"} \n"
        f"### Assistant: "
    )

In [9]:
data[0]  # Display the first two entries to verify the data structure

{'system_prompt': 'You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, \n    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, \n    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, \n    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by \n    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.\n    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],\n    v: ownship speed(m/s), \n    v_min: ownship minimum speed(m/s), \n    v_max: ownship maximum speed(m/s),\n    d

In [18]:
ex_prompt = format_prompt(data[100])
print(ex_prompt)  # Format the first entry to verify the prompt structure

### System:
You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, 
    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, 
    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, 
    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by 
    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.
    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],
    v: ownship speed(m/s), 
    v_min: ownship minimum speed(m/s), 
    v_max: ownship maximum speed(m/s),
    d1: the absolute 

In [13]:
inputs = tokenizer(ex_prompt, return_tensors="pt").to("cuda")
inputs

{'input_ids': tensor([[14374,   739,   510,  2610,   525,   458, 87182, 38647,   409,  6135,
         21242,   290, 17847,    13,  2411,  1817,   882,  3019,    11,   458,
          1828,  5270,  8315,   374, 29947,   264, 87838, 65176,    11,   715,
           262,  1741,   438, 53377,   476, 64257,  3501,    11,  1380,  1008,
         13009,   320, 58788,   661,   388,     8,   525, 29947,   438,  1632,
            13, 20205,   389,   279,  1995,   315,   279, 10582,   661,   388,
            11,   715,   262,   279,  1828,  5270,  1265,  1896,   458,  1917,
           311,  5648, 47353,    13,   576,  1828,  5270,  8315,  1172,   702,
          2615,   311,   279,  1995,   315,   279,  4065, 10582,   661,   388,
            11,   715,   262,   714,  1052,  2578,   387,  1008, 10582,   661,
           388,  4815,   279,  1828,  5270,    13,  4615,  3383,   374,   311,
          1492,   279,  1828,  5270, 14135,  5648, 47353,   448,  4065, 10582,
         32619, 14135,   553,   715,  

In [ ]:
with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
    output_tokens = base_model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False,
        temperature=0.0,
        top_p=0.95,
        # repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

response_text = tokenizer.batch_decode(output_tokens[0], skip_special_tokens=True)
# print("Full Response:")
response_text = "".join(response_text)
# print(response_text)
assistant_output = response_text.split("### Assistant:")[-1].strip()
print("Assistant's Response:")
print(assistant_output)

In [ ]:
# locate the word "Accelerate|Hold|Decelerate" in the response
match = re.search(r'\b(Accelerate|Hold|Decelerate)\b', assistant_output, re.IGNORECASE)
if match:
    predicted_action = match.group(1).capitalize()
    print(f"Predicted Action: {predicted_action}")

## 3. Accuracy or batch evaluation

In [14]:
# test data = randomly sample 1000 entries from data
import random
test_data = random.sample(ds_train, 100)
test_data

TypeError: Population must be a sequence.  For dicts or sets, use sorted(d).

In [16]:
def retrieve_action(response):
    match = re.search(r'\b(Accelerate|Hold|Decelerate)\b', response, re.IGNORECASE)
    if match:
        return match.group(1).capitalize()
    return None

In [15]:
test_data

[{'system_prompt': 'You are an airspace tactical deconfliction assistant. At each time step, an ownship agent is approaching a bottleneck waypoint, \n    such as merging or intersections points, where other agents (intruders) are approaching as well. Based on the information of the intruders, \n    the ownship should take an action to avoid collisions. The ownship agent only has access to the information of the front intruders, \n    but there might be other intruders behind the ownship. Your task is to help the ownship aircraft avoid collisions with front intruder aircraft by \n    suggesting appropriate speed adjustments. The ownship cannot unnecessarily decelerate since it might occlude the airspace for other agents behind it.\n    Now, given the ownship state: [v, v_min, v_max delta_d1, delta_d2, delta_dwpt1, delta_dwpt2, delta_v1, delta_v2, same_route1, same_route2],\n    v: ownship speed(m/s), \n    v_min: ownship minimum speed(m/s), \n    v_max: ownship maximum speed(m/s),\n    

In [22]:
def generate_response(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        output_tokens = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    decoded = tokenizer.batch_decode(output_tokens[0], skip_special_tokens=True)
    decoded = "".join(decoded)
    return decoded.split("### Assistant:")[-1].strip()

correct, i = 0, 0
for prompt, solution in zip(sampled_ds['prompt'], sampled_ds['solution']):
    pred_response = generate_response(prompt[0]['content'])
    print(pred_response)
    # the last word of the response should match "Acelerate|Hold|Decelerate"
    # so we can search for that word in the response and compare
    pred_action = retrieve_action(pred_response)
    actual_action = retrieve_action(solution)

    print(f"Actual action: {actual_action}, predicted action: {pred_action}.")

    if pred_action == actual_action:
        correct += 1
    i += 1

accuracy = correct / i
print(f"Accuracy over {i} samples: {accuracy:.4f}")

The recommended action is: Decelerate.
Actual action: Decelerate, predicted action: Decelerate.
The recommended action is: Decelerate.
Actual action: Hold, predicted action: Decelerate.
The recommended action is: Decelerate.
Actual action: Hold, predicted action: Decelerate.
The recommended action is: Decelerate.
Actual action: Accelerate, predicted action: Decelerate.
The recommended action is: Decelerate.
Actual action: Accelerate, predicted action: Decelerate.
The recommended action is: Decelerate.
Actual action: Hold, predicted action: Decelerate.
The recommended action is: Decelerate.
Actual action: Decelerate, predicted action: Decelerate.
The recommended action is: Decelerate.
Actual action: Accelerate, predicted action: Decelerate.
The recommended action is: Decelerate.
Actual action: Decelerate, predicted action: Decelerate.
The recommended action is: Decelerate.
Actual action: Decelerate, predicted action: Decelerate.
The recommended action is: Decelerate.
Actual action: Acce

In [21]:
def build_prompt(ex):
    return (
        f"### System:\n{ex['system_prompt'].strip()}\n"
        f"### User:\n{ex['user_prompt'].strip()}. Always start your response with: {"The recommended action is:"} \n"
        f"### Assistant: "
    )

def generate_batch_responses(examples):
    responses = []
    prompts = [build_prompt(example) for example in examples]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to("cuda")
    with sdpa_kernel(SDPBackend.FLASH_ATTENTION):
        output_tokens = base_model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            # top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
            # use_cache=True,
        )

    decoded = tokenizer.batch_decode(output_tokens[0], skip_special_tokens=True)
    # decoded = "".join(decoded)
    # assistant_response = decoded.split("### Assistant:")[-1].strip()
    # responses.append(assistant_response)
    return decoded

decoded = generate_batch_responses(test_data[:2])
# decoded
for i, response in enumerate(decoded):
    assistant_response = response.split("### Assistant:")[-1].strip()
    print(f"Response {i+1}: {assistant_response}")

Response 1: ###
Response 2: System
Response 3: :
Response 4: You
Response 5: are
Response 6: an
Response 7: airspace
Response 8: tactical
Response 9: de
Response 10: conf
Response 11: lict
Response 12: ion
Response 13: assistant
Response 14: .
Response 15: At
Response 16: each
Response 17: time
Response 18: step
Response 19: ,
Response 20: an
Response 21: own
Response 22: ship
Response 23: agent
Response 24: is
Response 25: approaching
Response 26: a
Response 27: bottleneck
Response 28: waypoint
Response 29: ,
Response 30: 
Response 31: 
Response 32: such
Response 33: as
Response 34: merging
Response 35: or
Response 36: intersections
Response 37: points
Response 38: ,
Response 39: where
Response 40: other
Response 41: agents
Response 42: (
Response 43: intr
Response 44: ud
Response 45: ers
Response 46: )
Response 47: are
Response 48: approaching
Response 49: as
Response 50: well
Response 51: .
Response 52: Based
Response 53: on
Response 54: the
Response 55: information
Response 56: of


In [ ]:
import math

def compute_perplexity(model, tokenizer, dataset):
    losses = []
    for ex in dataset:
        prompt = f"### System:\n{ex['system_prompt']}\n\n### User:\n{ex['user_prompt']}\n\n### Assistant:\n{ex['prompt_response']}"
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")
        with torch.no_grad():
            loss = model(**inputs, labels=inputs["input_ids"]).loss
        losses.append(loss.item())
    return math.exp(np.mean(losses))

perplexity = compute_perplexity(model, tokenizer, test_data)
print(f"Perplexity on test data: {perplexity:.4f}")

## Batched evaluation (not working)

In [ ]:
data_loaded = load_dataset('json', data_files=file_path, split='train').train_test_split(test_size=0.005, seed=42)
train_data = data_loaded['train']
test_data = data_loaded['test']

In [ ]:
test_data

In [ ]:
test_data[0:10]

In [ ]:
from torch.utils.data import DataLoader
BATCH_SIZE = 4
MAX_LEN = 1024


def collate_fn(examples):
    prompts = [build_prompt(ex) for ex in examples]
    # print(prompts[0])
    refs = [ex['prompt_response'] for ex in examples]
    encodings = tokenizer(prompts, return_tensors="pt", truncation=True, max_length=MAX_LEN, padding=True)
    encodings["prompts"] = prompts # keep raw text for decoding later
    encodings["refs"] = refs
    return encodings

dataloader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

In [ ]:
predictions, references = [], []

for batch in tqdm(dataloader, desc="Evaluating"):
    input_ids = batch['input_ids'].to("cuda")
    attention_mask = batch['attention_mask'].to("cuda")
    prompts = batch['prompts']
    refs = batch['refs']

    # print(batch)
    with torch.no_grad():
        output_tokens = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=64,
            do_sample=False,
            temperature=0.0,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.batch_decode(output_tokens, skip_special_tokens=True)
    for text, ex in zip(decoded, prompts):
        assistant_response = text.split("### Assistant:")[-1].strip()
        predictions.append(assistant_response)

    references.extend(refs)

In [ ]:
len(predictions), len(references)

In [ ]:
predictions[0:5], references[0:5]

In [ ]:
def accuracy_score(preds, refs):
    correct = sum(p.strip().lower() == r.strip().lower() for p, r in zip(preds, refs))
    return correct / len(refs)

acc = accuracy_score(predictions, references)
print(f"Batched Evaluation Accuracy: {acc:.4f}")